In [12]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
# import time
import imageio

In [13]:
def carregar_csv(caminho):
    df = pd.read_csv(caminho, header=None)

    # remove aspas + troca vírgula decimal por ponto
    df[0] = df[0].str.replace('"', '').str.replace(',', '.').astype(float)
    df[1] = df[1].str.replace('"', '').str.replace(',', '.').astype(float)
    df[2] = df[2].astype(int)

    dados = df[[0, 1]].values
    rotulos = df[2].values - 1

    return dados, rotulos

In [14]:
def gerar_dados():
    dados = []
    rotulos = []

    centros = [(-20, -20), (20, 20), (-20, 20), (20, -20)]

    for i, (cx, cy) in enumerate(centros):
        for _ in range(100):
            x = np.random.normal(cx, 2)
            y = np.random.normal(cy, 2)
            dados.append([x, y])
            rotulos.append(i)

    return np.array(dados), np.array(rotulos)

In [15]:
def normalizar(dados):
    min_vals = dados.min(axis=0)
    max_vals = dados.max(axis=0)
    return (dados - min_vals) / (max_vals - min_vals)

In [16]:
def distribuir_dados(grid, dados, rotulos):
    linhas, colunas = grid.shape
    indices = np.random.choice(linhas * colunas, len(dados), replace=False)
    xs, ys = np.unravel_index(indices, (linhas, colunas))

    for i in range(len(dados)):
        grid[xs[i], ys[i]] = (dados[i], rotulos[i])

In [17]:
def distancia(a, b):
    return np.linalg.norm(a - b)

In [18]:
class Ant:
    # def __init__(self, x, y, r=1, k1=0.1, k2=0.3):
    def __init__(self, x, y, r=1, k1=0.3, k2=0.6, alpha=0.1):

        self.x = x
        self.y = y
        self.r = r

        self.carregando = False 

        self.k1 = k1
        self.k2 = k2

        self.alpha = alpha

        self.item = None
        

    def move(self, max_x, max_y):
        direcoes = [(0, 1), (0, -1), (1, 0), (-1, 0)]
        dx, dy = random.choice(direcoes)
        
        self.x = (self.x + dx) % max_x
        self.y = (self.y + dy) % max_y


    def get_density(self, matrix):
        linhas, colunas = matrix.shape
        item_dados, item_rotulo = self.item if self.carregando else matrix[self.x, self.y]

        total_celulas = 0
        soma = 0

        for i in range(self.x - self.r, self.x + self.r + 1):
            for j in range(self.y - self.r, self.y + self.r + 1):
                ii = i % linhas
                jj = j % colunas
                total_celulas += 1

                if matrix[ii, jj] is not None:
                    if not self.carregando and ii == self.x and jj == self.y:
                        continue
                    vizinho_dados, vizinho_rotulo = matrix[ii, jj]
                    if vizinho_rotulo == item_rotulo:
                        soma += max(0, 1 - distancia(item_dados, vizinho_dados) / self.alpha)

        return soma / total_celulas if total_celulas > 0 else 0

    def decidir_acao(self, matrix):
        item_na_celula = matrix[self.x, self.y]

        # PEGAR
        if not self.carregando:
            if item_na_celula is not None:
                f = self.get_density(matrix)
                p_pegar = (1 - f) ** 2

                if random.random() < p_pegar:
                    self.item = item_na_celula
                    matrix[self.x, self.y] = None
                    self.carregando = True

        else:
            if item_na_celula is None:
                f = self.get_density(matrix)
                p_largar = f ** 2

                if random.random() < p_largar:
                    matrix[self.x, self.y] = self.item
                    self.item = None
                    self.carregando = False

In [19]:
def visualizar(grid):
    plt.figure(figsize=(6,6))
    cores = ['red', 'blue', 'green', 'purple']

    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            if grid[i, j] is not None:
                _, label = grid[i, j]
                plt.scatter(i, j, color=cores[label], s=10)

    plt.title("Clusters formados")
    plt.show()

In [20]:
def renderizar_grid(grid, scale=6):
    img = np.zeros((grid.shape[0], grid.shape[1], 3), dtype=np.uint8)

    cores = [
        [255, 0, 0],    # vermelho
        [0, 0, 255],    # azul
        [0, 255, 0],    # verde
        [128, 0, 128]   # roxo
    ]

    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            if grid[i, j] is not None:
                _, label = grid[i, j]
                img[i, j] = cores[label]

    # aumenta resolução
    img = np.kron(img, np.ones((scale, scale, 1)))

    return img.astype(np.uint8)

In [21]:
# dados, rotulos = carregar_csv("Base.csv")

# print(dados.shape)
# print(rotulos.shape)

In [22]:
LARGURA = 64
ALTURA = 64
NUM_FORMIGAS = 100
MAX_INTERACOES = 1000000

# gerar dados
# dados, rotulos = gerar_dados()
# dados, rotulos = carregar_csv("Base.csv")
dados, rotulos = carregar_csv("Base.csv")
# print(len(dados))
dados = normalizar(dados)

# criar grid
grid = np.empty((LARGURA, ALTURA), dtype=object)
grid[:] = None

distribuir_dados(grid, dados, rotulos)

# criar formigas
formigas = [
    Ant(random.randint(0, LARGURA-1),
        random.randint(0, ALTURA-1),
        r=1, k1=0.01, k2=0.015, alpha=30)
    for _ in range(NUM_FORMIGAS)
]

print("Simulação iniciando...")

frame_inicial = renderizar_grid(grid)
imageio.imwrite("estado_inicial.png", frame_inicial)

with imageio.get_writer("ant_clustering.gif", mode="I", fps=5) as writer:

    for it in range(MAX_INTERACOES):
        random.shuffle(formigas)

        for ant in formigas:
            ant.move(LARGURA, ALTURA)
            ant.decidir_acao(grid)

        if it % 10000 == 0:
            print(f"Frame {it}")
            frame = renderizar_grid(grid)
            writer.append_data(frame)

frame_final = renderizar_grid(grid)
imageio.imwrite("estado_final.png", frame_final)

print("Simulação finalizada!")


Simulação iniciando...
Frame 0
Frame 10000
Frame 20000
Frame 30000
Frame 40000
Frame 50000
Frame 60000
Frame 70000
Frame 80000
Frame 90000
Frame 100000
Frame 110000
Frame 120000
Frame 130000
Frame 140000
Frame 150000
Frame 160000
Frame 170000
Frame 180000
Frame 190000
Frame 200000
Frame 210000
Frame 220000
Frame 230000
Frame 240000
Frame 250000
Frame 260000
Frame 270000
Frame 280000
Frame 290000
Frame 300000
Frame 310000
Frame 320000
Frame 330000
Frame 340000
Frame 350000
Frame 360000
Frame 370000
Frame 380000
Frame 390000
Frame 400000
Frame 410000
Frame 420000
Frame 430000
Frame 440000
Frame 450000
Frame 460000
Frame 470000
Frame 480000
Frame 490000
Frame 500000
Frame 510000
Frame 520000
Frame 530000
Frame 540000
Frame 550000
Frame 560000
Frame 570000
Frame 580000
Frame 590000
Frame 600000
Frame 610000
Frame 620000
Frame 630000
Frame 640000
Frame 650000
Frame 660000
Frame 670000
Frame 680000
Frame 690000
Frame 700000
Frame 710000
Frame 720000
Frame 730000
Frame 740000
Frame 750000
Fra